Sources
https://aboskovic21.github.io/projects/pu_gb.pdf
optuna

In [11]:


import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams['text.usetex'] = False

In [12]:
# Main code voor processen van de data
import uproot
import sklearn
import xgboost
import pathlib
import matplotlib.pyplot as plt
import scienceplots
import numpy as np
from matplotlib.pyplot import figure
import corner

figure(figsize=(8, 6), dpi=80)

# Background
with uproot.open(pathlib.Path(r"Training_Data\data.root")) as file:
    tree = file['treeMLDplus']
    tree.show()
    branches = tree.keys()
    print(branches)
    training_data_bkg = tree.arrays(branches, library="np")
    training_data_bkg['species'] = np.zeros(len(training_data_bkg['inv_mass']))
    # data_matrix = np.column_stack([data[b][:10000] for b in branches])

    # plt.figure(dpi=300)
    # fig = corner.corner(data_matrix, labels=branches)
    # inv_mass_training = tree.arrays(['inv_mass'],library='np')

# Signal
with uproot.open(pathlib.Path(r"Training_Data\FD.root")) as file:
    tree = file['treeMLDplus']
    tree.show()
    branches = tree.keys()
    print(branches)
    training_data_FD = tree.arrays(branches, library="np")
    training_data_FD['species'] = np.ones(len(training_data_FD['inv_mass']))
    # data_matrix = np.column_stack([data[b][:10000] for b in branches])

    # plt.figure(dpi=300)
    # fig = corner.corner(data_matrix, labels=branches)
    # inv_mass_FD = tree.arrays(['inv_mass'],library='np')


name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
inv_mass             | float                    | AsDtype('>f4')
pt_cand              | float                    | AsDtype('>f4')
d_len                | float                    | AsDtype('>f4')
d_len_xy             | float                    | AsDtype('>f4')
norm_dl_xy           | float                    | AsDtype('>f4')
cos_p                | float                    | AsDtype('>f4')
cos_p_xy             | float                    | AsDtype('>f4')
imp_par_xy           | float                    | AsDtype('>f4')
sig_vert             | float                    | AsDtype('>f4')
max_norm_d0d0exp     | float                    | AsDtype('>f4')
nsigComb_Pi_0        | float                    | AsDtype('>f4')
nsigComb_K_0         | float                    | AsDtype('>f4')
nsigComb_Pi_1        | float                    | AsDtype(

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

kbg_df = pd.DataFrame(training_data_FD)
FD_df = pd.DataFrame(training_data_bkg)

full_data = pd.concat([kbg_df,FD_df])


In [14]:
x_data = full_data[branches]
y_data = full_data['species']
print(set(y_data))


x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=7)
print(np.unique(y_train, return_counts=True))

# Settings we train the xgbclassifier with, the options we pass to the xgbclassifier are the hyperparameters we want to optimise.
pos_class_weight = len(training_data_bkg['inv_mass']) / len(training_data_FD['inv_mass'])
model = xgb.XGBClassifier(
    n_estimators=100,
    objective='binary:logistic',
    scale_pos_weight=pos_class_weight,
    max_delta_step=1,
    random_state=42
)
model.fit(x_train, y_train)

{0.0, 1.0}
(array([0., 1.]), array([3741792,    4560]))


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [15]:
from sklearn.metrics import recall_score, precision_score, roc_auc_score, accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
# Define a function to evaluate the results
def evaluate_results(y_test, y_predict):
    print('Classification results:')
    f1 = f1_score(y_test, y_predict)
    print("f1: %.2f%%" % (f1 * 100.0)) 
    roc = roc_auc_score(y_test, y_predict)
    print("roc: %.2f%%" % (roc * 100.0)) 
    rec = recall_score(y_test, y_predict, average='binary')
    print("recall: %.2f%%" % (rec * 100.0)) 
    prc = precision_score(y_test, y_predict, average='binary')
    print("precision: %.2f%%" % (prc * 100.0))

# Evaluate the model
predictions = model.predict(x_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))

# Make predictions on the testing set and evaluate the results
y_predict = model.predict(x_test)
evaluate_results(y_test, y_predict)

Confusion Matrix:
[[928095   7281]
 [    84   1128]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00    935376
         1.0       0.13      0.93      0.23      1212

    accuracy                           0.99    936588
   macro avg       0.57      0.96      0.62    936588
weighted avg       1.00      0.99      1.00    936588

Classification results:
f1: 23.45%
roc: 96.15%
recall: 93.07%
precision: 13.41%


In [28]:
import optuna
# Define an objective function. This will find how good the model is performing, based on different hyperparameters.
# Using this function optuna will sample the hyperparameterspace to find the best hyperparameters for training.
def objective(trial):
    data, target =  full_data[branches], full_data['species']
    train_x, test_x, train_y, test_y = train_test_split(data, target, test_size=0.25)
    dtrain = xgb.DMatrix(train_x, label=train_y)
    dtest = xgb.DMatrix(test_x, label=test_y)

    param = {
        "silent": 1,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0,log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0,log=True),
    }

    if param["booster"] == "gbtree" or param["booster"] == "dart":
        param["max_depth"] = trial.suggest_int("max_depth", 1, 9)
        param["eta"] = trial.suggest_float("eta", 1e-8, 1.0,log=True)
        param["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0,log=True)
        param["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])
    if param["booster"] == "dart":
        param["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
        param["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
        param["rate_drop"] = trial.suggest_float("rate_drop", 1e-8, 1.0,log=True)
        param["skip_drop"] = trial.suggest_float("skip_drop", 1e-8, 1.0,log=True)

    # Add a callback for pruning.
    # We are sampling the hyperparameterspace for what combination of hyperparameters results in a fast learning algorithm
    # For samples that learn very slowly, we do not need to train them for many epochs
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "validation-auc")
    bst = xgb.train(param, dtrain, evals=[(dtest, "validation")], callbacks=[pruning_callback])
    preds = bst.predict(dtest)
    pred_labels = np.rint(preds)
    accuracy = sklearn.metrics.f1_score(test_y, pred_labels) # F1 score voor ongebalanceerde data
    # Let op met welke metric we gebruiken als accuracy hier. Omdat de dataset heel ongebalanceerd is, 
    # moedigen we hier modellen aan die alles klassificeren als geen D+ deeltje, omdat dat al best goed in de buurt van het juiste antwoord is
    # We hebben immers veel minder D+ deeltjes
    return accuracy

study = optuna.create_study()
study.optimize(objective, n_trials=30)
print(study.best_trial)

[I 2025-11-19 19:43:07,306] A new study created in memory with name: no-name-7a32ddab-b1ee-4a76-9341-857c2fb4645c
c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\xgboost\training.py:183: UserWarning:

[19:43:09] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "silent" } are not used.




[0]	validation-auc:0.98288
[1]	validation-auc:0.98552
[2]	validation-auc:0.98682
[3]	validation-auc:0.98680
[4]	validation-auc:0.98692
[5]	validation-auc:0.98697
[6]	validation-auc:0.98748
[7]	validation-auc:0.98764
[8]	validation-auc:0.98770
[9]	validation-auc:0.98777


[I 2025-11-19 19:43:19,087] Trial 0 finished with value: 0.0 and parameters: {'booster': 'dart', 'lambda': 0.0025154730321837715, 'alpha': 2.0917744716418024e-07, 'max_depth': 7, 'eta': 0.005176870315581991, 'gamma': 1.3913628009809064e-06, 'grow_policy': 'depthwise', 'sample_type': 'weighted', 'normalize_type': 'tree', 'rate_drop': 0.0012819241966547069, 'skip_drop': 0.26781109209332205}. Best is trial 0 with value: 0.0.
c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\xgboost\training.py:183: UserWarning:

[19:43:21] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "silent" } are not used.




[0]	validation-auc:0.93070
[1]	validation-auc:0.92667
[2]	validation-auc:0.92541
[3]	validation-auc:0.92466
[4]	validation-auc:0.92408
[5]	validation-auc:0.92360
[6]	validation-auc:0.92320
[7]	validation-auc:0.92287
[8]	validation-auc:0.92261
[9]	validation-auc:0.92240


[I 2025-11-19 19:43:26,531] Trial 1 finished with value: 0.0 and parameters: {'booster': 'gblinear', 'lambda': 0.2786651707073718, 'alpha': 2.5051492449269025e-06}. Best is trial 0 with value: 0.0.
c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\xgboost\training.py:183: UserWarning:

[19:43:28] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "silent" } are not used.




...

[0]	validation-auc:0.78749
[1]	validation-auc:0.78749
[2]	validation-auc:0.78749
[3]	validation-auc:0.78749
[4]	validation-auc:0.78749
[5]	validation-auc:0.78749
[6]	validation-auc:0.78749
[7]	validation-auc:0.78749
[8]	validation-auc:0.78749
[9]	validation-auc:0.78749


[I 2025-11-19 19:49:10,424] Trial 28 finished with value: 0.0 and parameters: {'booster': 'gbtree', 'lambda': 0.0003565131003032401, 'alpha': 5.3255979199324594e-08, 'max_depth': 2, 'eta': 5.450879463560828e-05, 'gamma': 1.8336989177759126e-07, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.0.
c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\xgboost\training.py:183: UserWarning:

[19:49:15] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "silent" } are not used.




[0]	validation-auc:0.94248


[I 2025-11-19 19:49:17,661] Trial 29 pruned. Trial was pruned at iteration 0.


FrozenTrial(number=0, state=1, values=[0.0], datetime_start=datetime.datetime(2025, 11, 19, 19, 43, 7, 307487), datetime_complete=datetime.datetime(2025, 11, 19, 19, 43, 19, 87491), params={'booster': 'dart', 'lambda': 0.0025154730321837715, 'alpha': 2.0917744716418024e-07, 'max_depth': 7, 'eta': 0.005176870315581991, 'gamma': 1.3913628009809064e-06, 'grow_policy': 'depthwise', 'sample_type': 'weighted', 'normalize_type': 'tree', 'rate_drop': 0.0012819241966547069, 'skip_drop': 0.26781109209332205}, user_attrs={}, system_attrs={}, intermediate_values={0: 0.9828849041103417, 1: 0.98552043034651, 2: 0.9868188546587265, 3: 0.9867978532972667, 4: 0.9869241166388159, 5: 0.9869700931827411, 6: 0.9874778695572032, 7: 0.9876423502673275, 8: 0.9876963230992181, 9: 0.987773482573699}, distributions={'booster': CategoricalDistribution(choices=('gbtree', 'gblinear', 'dart')), 'lambda': FloatDistribution(high=1.0, log=True, low=1e-08, step=None), 'alpha': FloatDistribution(high=1.0, log=True, low=1

In [17]:
import plotly
df = study.trials_dataframe()

best = study.best_trial

data = {
    "value": best.value,
    "params": best.params,
    "user_attrs": best.user_attrs,
    "system_attrs": best.system_attrs
}

objective(best)

C:\Users\sande\AppData\Local\Temp\ipykernel_20096\1595713359.py:15: FutureWarning:

suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.

C:\Users\sande\AppData\Local\Temp\ipykernel_20096\1595713359.py:16: FutureWarning:

suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.

c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\xgboost\training.py:183: UserWarning:

[19:27:05] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "silent" } are not used.




[0]	validation-auc:0.94531
[1]	validation-auc:0.94768
[2]	validation-auc:0.94682
[3]	validation-auc:0.94564
[4]	validation-auc:0.94433
[5]	validation-auc:0.94337
[6]	validation-auc:0.94266
[7]	validation-auc:0.94184
[8]	validation-auc:0.94107
[9]	validation-auc:0.94036


0.9987076494680692

In [18]:
import optuna
import optuna_integration
from plotly.io import show
# https://optuna.readthedocs.io/en/stable/reference/visualization/index.html

fig = optuna.visualization.plot_param_importances(study)
show(fig)
fig = optuna.visualization.plot_intermediate_values(study)
show(fig)
fig = optuna.visualization.plot_timeline(study)
show(fig)

In [ ]:

best_params = study.best_params
model = xgb.XGBClassifier(**best_params
)
model.fit(x_train, y_train)
pred = model.predict(x_test)
print(classification_report(y_test, pred))
evaluate_results(y_test, pred)
#https://www.geeksforgeeks.org/machine-learning/handling-imbalanced-data-for-classification/

c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    935376
         1.0       0.00      0.00      0.00      1212

    accuracy                           1.00    936588
   macro avg       0.50      0.50      0.50    936588
weighted avg       1.00      1.00      1.00    936588

Classification results:
f1: 0.00%
roc: 50.00%
recall: 0.00%
precision: 0.00%


c:\Users\sande\anaconda3\envs\CAML\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

